In [1]:
import sys
sys.path.append("/opt/devel/rbyrne/rlb_LWA/LWA_data_preprocessing")
from LWA_calibrate import *
from calico import calibration_wrappers, caldata, calibration_optimization
import pyuvdata
import os
import numpy as np
import sys
import scipy

/opt/devel/rbyrne/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-03 17:57:06,945	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# Run aoflagger
aoflagger_strategy_file = "/lustre/ghellbourg/AOFlagger_strat_opt/LWA_opt_GH1.lua"
use_datafile_path = "/fast/rbyrne/20260419_055641-055832_44MHz.ms"
subprocess.run(["aoquality", "remove", use_datafile_path], check=True)
subprocess.run(["aoquality", "collect", use_datafile_path], check=True)
subprocess.run(
    ["aoflagger", "--strategy", aoflagger_strategy_file, use_datafile_path],
    check=True,
)

# Re-image data to make sure clean components are properly created
#image_data(
#    "/lustre/rbyrne/2026-04-19/20260419_055641-055832_44MHz_17h_cal_calibrated",
#    "/lustre/rbyrne/2026-04-19/20260419_055641-055832_44MHz_17h_cal_calibrated.ms",
#    niter=5000,
#)

data = pyuvdata.UVData()
data.read(use_datafile_path, data_column="DATA")

model = pyuvdata.UVData()
model.read(
    "/lustre/rbyrne/2026-04-19/20260419_055641-055832_44MHz_17h_cal_calibrated.ms",
    data_column="MODEL_DATA",
)

flag_antenna_list = [
    "LWA-266B",
    "LWA-269B",
    "LWA-282B",
    "LWA-310B",
    "LWA-341A",
    "LWA-341B",
    "LWA-255B",
    "LWA-263A",
    "LWA-263B",
    "LWA-272A",
    "LWA-272B",
    "LWA-283A",
    "LWA-280A",
    "LWA-280B",
    "LWA-288B",
    "LWA-284A",
    "LWA-292B",
    "LWA-336A",
    "LWA-336B",
    "LWA-335A",
    "LWA-004B",
    "LWA-005B",
    "LWA-040A",
    "LWA-302A",
    "LWA-302B",
    "LWA-013B",
    "LWA-016B",
    "LWA-015A",
    "LWA-015B",
    "LWA-014A",
    "LWA-014B",
    "LWA-025A",
    "LWA-348B",
    "LWA-365A",
    "LWA-365B",
    "LWA-364A",
    "LWA-364B",
    "LWA-086A",
    "LWA-086B",
    "LWA-122A",
    "LWA-334A",
    "LWA-361A",
    "LWA-068B",
    "LWA-069A",
    "LWA-252A",
    "LWA-252B",
    "LWA-289A",
    "LWA-299B",
    "LWA-107A",
    "LWA-110A",
    "LWA-112A",
    "LWA-112B",
    "LWA-111A",
    "LWA-143B",
    "LWA-124A",
    "LWA-159A",
    "LWA-224A",
    "LWA-347A",
    "LWA-167B",
    "LWA-227B",
    "LWA-329A",
    "LWA-234A",
    "LWA-206A",
    "LWA-206B",
    "LWA-210A",
    "LWA-235B",
    "LWA-342B",
    "LWA-214B",
    "LWA-221B",
    "LWA-315A",
]
flag_antennas(
    data,
    antenna_names=flag_antenna_list,
    inplace=True,
)

0....10....20....30....40....50....60....70....80....90....100
Writing quality tables...
Done.
AOFlagger 3.2 (2022-05-19) command line application
Author: André Offringa (offringa@gmail.com)

Starting strategy on 2026-Aug-03 17:58:13.721884


In [3]:
uvcal = calibration_wrappers.sky_based_calibration_wrapper(
    data,
    model,
    min_cal_baseline_lambda=10,
    max_cal_baseline_lambda=125,
    gains_multiply_model=True,
    verbose=True,
    get_crosspol_phase=False,
    xtol=1e-6,
    maxiter=200,
    antenna_flagging_iterations=0,
    parallel=False,
    lambda_val=0,
)

The entry name zenith_at_jd2461149.748347 is not unique inside the phase center catalog, adding anyways.


Formatting data...


divide by zero encountered in divide
invalid value encountered in divide


Done. Data formatting time 2.560772454738617 minutes.
Running calibration optimization...
Optimization terminated successfully.
         Current function value: 586.744208
         Iterations: 9
         Function evaluations: 12
         Gradient evaluations: 12
         Hessian evaluations: 9
Optimization terminated successfully.
Freq. 0 Pol. 0, optimization time: 0.036817570527394615 minutes
Optimization terminated successfully.
         Current function value: 605.171943
         Iterations: 9
         Function evaluations: 12
         Gradient evaluations: 12
         Hessian evaluations: 9
Optimization terminated successfully.
Freq. 0 Pol. 1, optimization time: 0.03643338680267334 minutes
Optimization terminated successfully.
         Current function value: 575.954755
         Iterations: 9
         Function evaluations: 12
         Gradient evaluations: 12
         Hessian evaluations: 9
Optimization terminated successfully.
Freq. 1 Pol. 0, optimization time: 0.03634266455968221

KeyboardInterrupt: 

In [ ]:
np.max(model.data_array)

np.complex64(5371.5366+409.1149j)

In [ ]:
caldata_obj = caldata.CalData()

In [ ]:
caldata_obj.load_data(
    data=data,
    model=model,
    gain_init_calfile=None,
    gain_init_to_vis_ratio=True,
    gains_multiply_model=True,
    gain_init_stddev=0,
    check_vis_ordering=True,
    N_feed_pols=2,
    min_cal_baseline_lambda=10,
    max_cal_baseline_lambda=125,
    lambda_val=0,
)

divide by zero encountered in divide
invalid value encountered in divide
divide by zero encountered in scalar divide


In [ ]:
np.max(caldata_obj.model_visibilities)

np.complex128(5121.69287109375+349.6324157714844j)

In [ ]:
np.max(calibration_optimization.hessian_skycal_wrapper(
    np.ones(2*caldata_obj.Nants),
    caldata_obj,
    np.arange(caldata_obj.Nants),
    0,
    0,
))

np.float64(160504112695.76816)

In [ ]:
caldata_obj.sky_based_calibration(
    xtol=1e-6,
    maxiter=100,
    get_crosspol_phase=False,
    parallel=False,
    verbose=True,
)

invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply


         Current function value: nan
         Iterations: 0
         Function evaluations: 1
         Gradient evaluations: 1
         Hessian evaluations: 1
Freq. 0 Pol. 0, optimization time: 0.01396398146947225 minutes


invalid value encountered in multiply


         Current function value: nan
         Iterations: 0
         Function evaluations: 1
         Gradient evaluations: 1
         Hessian evaluations: 1
Freq. 0 Pol. 1, optimization time: 0.014182929197947185 minutes
         Current function value: nan
         Iterations: 0
         Function evaluations: 1
         Gradient evaluations: 1
         Hessian evaluations: 1
Freq. 1 Pol. 0, optimization time: 0.014457714557647706 minutes
         Current function value: nan
         Iterations: 0
         Function evaluations: 1
         Gradient evaluations: 1
         Hessian evaluations: 1
Freq. 1 Pol. 1, optimization time: 0.014435684680938721 minutes


KeyboardInterrupt: 

In [ ]:
freq_ind = 0
verbose=True
xtol=1e-5
maxiter=200

gains_fit = np.full(
    (caldata_obj.Nants, caldata_obj.N_feed_pols),
    np.nan + 1j * np.nan,
    dtype=complex,
)
if np.max(caldata_obj.visibility_weights[:, :, freq_ind, :]) == 0.0:
    print("ERROR: All data flagged.")
    gains_fit[:, :] = np.nan + 1j * np.nan

for feed_pol_ind, feed_pol in enumerate(caldata_obj.feed_polarization_array):
    vis_pol_ind = np.where(caldata_obj.vis_polarization_array == feed_pol)[0]

    if (
        np.max(caldata_obj.visibility_weights[:, :, freq_ind, vis_pol_ind]) == 0.0
    ):  # All flagged
        gains_fit[:, feed_pol_ind] = np.nan + 1j * np.nan
    else:
        vis_weights_summed = np.sum(
            caldata_obj.visibility_weights[:, :, freq_ind, feed_pol_ind], axis=0
        )  # Sum over times
        weight_per_ant = np.bincount(
            caldata_obj.ant1_inds,
            weights=vis_weights_summed,
            minlength=caldata_obj.Nants,
        ) + np.bincount(
            caldata_obj.ant2_inds,
            weights=vis_weights_summed,
            minlength=caldata_obj.Nants,
        )
        ant_inds = np.where(weight_per_ant > 0.0)[0]

        gains_init_flattened = np.stack(
            (
                np.real(caldata_obj.gains[ant_inds, freq_ind, feed_pol_ind]),
                np.imag(caldata_obj.gains[ant_inds, freq_ind, feed_pol_ind]),
            ),
            axis=1,
        ).flatten()

        # Minimize the cost function
        result = scipy.optimize.minimize(
            calibration_optimization.cost_skycal_wrapper,
            gains_init_flattened,
            args=(caldata_obj, ant_inds, freq_ind, vis_pol_ind),
            method="Newton-CG",
            jac=calibration_optimization.jacobian_skycal_wrapper,
            hess=calibration_optimization.hessian_skycal_wrapper,
            options={"disp": verbose, "xtol": xtol, "maxiter": maxiter},
        )
        if verbose:
            print(result.message)
        sys.stdout.flush()
        gains_fit_single_pol = np.reshape(result.x, (len(ant_inds), 2))
        gains_fit[ant_inds, feed_pol_ind] = (
            gains_fit_single_pol[:, 0] + 1j * gains_fit_single_pol[:, 1]
        )

        # Ensure that the phase of the gains is mean-zero
        # This adds should be handled by the phase regularization term, but
        # this step removes any optimizer precision effects.
        avg_angle = np.arctan2(
            np.nanmean(np.sin(np.angle(gains_fit[:, feed_pol_ind]))),
            np.nanmean(np.cos(np.angle(gains_fit[:, feed_pol_ind]))),
        )
        gains_fit[:, feed_pol_ind] *= np.cos(avg_angle) - 1j * np.sin(avg_angle)

         Current function value: nan
         Iterations: 0
         Function evaluations: 2
         Gradient evaluations: 2
         Hessian evaluations: 1


NameError: name 'end_optimize' is not defined

In [ ]:
caldata_obj.gains

array([[[inf+nanj, nan+nanj],
        [inf+nanj, nan+nanj],
        [inf +0.j, inf +0.j],
        ...,
        [inf +0.j, inf +0.j],
        [inf +0.j, inf +0.j],
        [inf +0.j, inf +0.j]],

       [[inf+nanj, inf+nanj],
        [inf+nanj, inf+nanj],
        [inf +0.j, inf +0.j],
        ...,
        [inf +0.j, inf +0.j],
        [inf +0.j, inf +0.j],
        [inf +0.j, inf +0.j]],

       [[inf+nanj, inf+nanj],
        [inf+nanj, inf+nanj],
        [inf +0.j, inf +0.j],
        ...,
        [inf +0.j, inf +0.j],
        [inf +0.j, inf +0.j],
        [inf +0.j, inf +0.j]],

       ...,

       [[inf+nanj, inf+nanj],
        [inf+nanj, inf+nanj],
        [inf +0.j, inf +0.j],
        ...,
        [inf +0.j, inf +0.j],
        [inf +0.j, inf +0.j],
        [inf +0.j, inf +0.j]],

       [[nan+nanj, inf+nanj],
        [nan+nanj, inf+nanj],
        [inf +0.j, inf +0.j],
        ...,
        [inf +0.j, inf +0.j],
        [inf +0.j, inf +0.j],
        [inf +0.j, inf +0.j]],

       [[inf

In [ ]:
vis_amp_ratio = np.abs(caldata_obj.model_visibilities) / np.abs(caldata_obj.data_visibilities)

divide by zero encountered in divide
invalid value encountered in divide


In [ ]:
vis_amp_ratio[np.where(caldata_obj.data_visibilities == 0.0)] = np.nan

In [ ]:
1 / np.sqrt(np.nanmedian(vis_amp_ratio))

divide by zero encountered in scalar divide


np.float64(inf)

In [ ]:
np.nanmedian(vis_amp_ratio)

np.float64(0.0)

In [ ]:
vis_amp_ratio

array([[[[   38378.50181345,  2325198.17369135,        0.        ,
                 0.        ],
         [   42566.75595016,  6487880.51012002,        0.        ,
                 0.        ],
         [   53393.69271284,  1832887.0466424 ,        0.        ,
                 0.        ],
         ...,
         [  166057.09729496,  8240938.4202084 ,        0.        ,
                 0.        ],
         [  157545.49755375, 10786953.61321162,        0.        ,
                 0.        ],
         [  178379.01587524,  3347417.33464688,        0.        ,
                 0.        ]],

        [[   60827.72461407,  2087873.57663057,        0.        ,
                 0.        ],
         [   36710.44504605,  1259827.61761753,        0.        ,
                 0.        ],
         [   83683.81015516,  4879320.45524052,        0.        ,
                 0.        ],
         ...,
         [   56296.20538668,  5847859.86019062,        0.        ,
                 0.        ],
